In [ ]:
suppressPackageStartupMessages({
  library(dplyr)
  library(tibble)
  library(tidyr)
  library(lme4)
  library(ggplot2)
  library(parallel)
})

## Fit multiple outcomes

In [ ]:
setwd("/sharedscratch/maff1/md4002-elena-adir")
source("scripts/R/loo_lmer.R")

In [ ]:
outcomes <- c("TAU_capped", "PTAU_capped", "ABETA_capped")

all_res <- lapply(outcomes, function(outcome) {

  formula_text <- paste0(
    outcome,
    " ~ VISCODEnum_z * GSK3B_capped + AGE_z + PTGENDER + DIAGNOSIS_bl + APOE4 + PTEDUCAT_z + BMI_capped +(VISCODEnum_z | RID)"
  )
  
  if(!dir.exists("results/sensitivity")) {dir.create("results/sensitivity")}
     
  out_file <- paste0("sensitivity_", tolower(outcome), "_loo.csv")
  tmp <- run_loo_lmer(
    input_csv = "data/tmp/063_Phase_II_Cohort",
    output_csv = file.path("results/sensitivity", out_file),
    formula_text = formula_text,
    term_of_interest = "GSK3B_capped",
    id_var = "RID",
    use_parallel = TRUE
  )

  tmp$loo_results$outcome <- outcome
  tmp$full_model_metrics$outcome <- outcome
  tmp
})

loo_all <- dplyr::bind_rows(lapply(all_res, `[[`, "loo_results"))
metrics_all <- dplyr::bind_rows(lapply(all_res, `[[`, "full_model_metrics"))

files <- c(
  loo_all = "results/sensitivity/loo_all.rds",
  metrics_all = "results/sensitivity/metrics_all.rds"
)

for (nm in names(files)) {
  if (!file.exists(files[[nm]])) {
    saveRDS(get(nm), files[[nm]])
  }
}

# performance metrics dictionary

In [ ]:
library(knitr)

dict <- data.frame(
  Variable = c(
    "subject","n_rows_removed","outcome",
    "beta_full","se_full","beta_loo","delta_beta","pct_change_beta","dfbetas_term",
    "logLik_full","logLik_loo","delta_logLik","likelihood_distance",
    "AIC_full","AIC_loo","delta_AIC","BIC_full","BIC_loo","delta_BIC",
    "sigma_full","sigma_loo","delta_sigma",
    "R2_marginal_full","R2_marginal_loo","delta_R2_marginal",
    "R2_conditional_full","R2_conditional_loo","delta_R2_conditional",
    "ICC_full","ICC_loo","delta_ICC",
    "RMSE_full","RMSE_loo","delta_RMSE",
    "influential_dfbetas","influential_likelihood","influential_aic","fit_failed"
  ),
  Description = c(
    "Subject ID removed in LOO",
    "Number of observations removed",
    "Model outcome variable",
    
    "Full-model coefficient (term of interest)",
    "Standard error (full model)",
    "Coefficient after removing subject",
    "Change in coefficient (full - LOO)",
    "Percentage change in coefficient",
    "Standardised influence (delta beta / SE)",
    
    "Full-model log-likelihood",
    "LOO log-likelihood",
    "Change in log-likelihood",
    "2 * delta logLik",
    
    "Full-model AIC",
    "LOO AIC",
    "Change in AIC (LOO - full)",
    "Full-model BIC",
    "LOO BIC",
    "Change in BIC",
    
    "Residual standard deviation (full model)",
    "Residual SD (LOO)",
    "Change in residual SD",
    
    "Marginal R2 (fixed effects, full)",
    "Marginal R2 (LOO)",
    "Change in marginal R2",
    
    "Conditional R2 (full model)",
    "Conditional R2 (LOO)",
    "Change in conditional R2",
    
    "Intraclass correlation (full model)",
    "Intraclass correlation (LOO)",
    "Change in ICC",
    
    "Root Mean Squared Error (full)",
    "RMSE (LOO)",
    "Change in RMSE",
    
    "Influential by DFBETAS threshold",
    "Influential by likelihood test",
    "Influential if delta AIC < -2",
    "Model failed to converge"
  ),
  stringsAsFactors = FALSE
)

kable(dict, escape=FALSE)

|Variable               |Description                               |
|:----------------------|:-----------------------------------------|
|subject                |Subject ID removed in LOO                 |
|n_rows_removed         |Number of observations removed            |
|outcome                |Model outcome variable                    |
|beta_full              |Full-model coefficient (term of interest) |
|se_full                |Standard error (full model)               |
|beta_loo               |Coefficient after removing subject        |
|delta_beta             |Change in coefficient (full - LOO)        |
|pct_change_beta        |Percentage change in coefficient          |
|dfbetas_term           |Standardised influence (delta beta / SE)  |
|logLik_full            |Full-model log-likelihood                 |
|logLik_loo             |LOO log-likelihood                        |
|delta_logLik           |Change in log-likelihood                  |
|likelihood_distance    |2 * delta logLik                          |
|AIC_full               |Full-model AIC                            |
|AIC_loo                |LOO AIC                                   |
|delta_AIC              |Change in AIC (LOO - full)                |
|BIC_full               |Full-model BIC                            |
|BIC_loo                |LOO BIC                                   |
|delta_BIC              |Change in BIC                             |
|sigma_full             |Residual standard deviation (full model)  |
|sigma_loo              |Residual SD (LOO)                         |
|delta_sigma            |Change in residual SD                     |
|R2_marginal_full       |Marginal R2 (fixed effects, full)         |
|R2_marginal_loo        |Marginal R2 (LOO)                         |
|delta_R2_marginal      |Change in marginal R2                     |
|R2_conditional_full    |Conditional R2 (full model)               |
|R2_conditional_loo     |Conditional R2 (LOO)                      |
|delta_R2_conditional   |Change in conditional R2                  |
|ICC_full               |Intraclass correlation (full model)       |
|ICC_loo                |Intraclass correlation (LOO)              |
|delta_ICC              |Change in ICC                             |
|RMSE_full              |Root Mean Squared Error (full)            |
|RMSE_loo               |RMSE (LOO)                                |
|delta_RMSE             |Change in RMSE                            |
|influential_dfbetas    |Influential by DFBETAS threshold          |
|influential_likelihood |Influential by likelihood test            |
|influential_aic        |Influential if delta AIC < -2             |
|fit_failed             |Model failed to converge                  |

## Plots

In [ ]:
source("scripts/R/loo_lmer_plots.R")

In [ ]:
plots <- make_all_plots(metrics_all, loo_all)

In [ ]:
plots$loo_summary_table

In [ ]:
library("cowplot")

p_sensi <- plot_grid(plots$full_model_metrics, plots$loo_distributions, 
          plots$robustness_summary, plots$influential_counts, 
          ncol = 2, 
          labels = c("A", "B", "C", "D")
)
ggsave("results/sensitivity/loo_panel.png", p_sensi, width = 10, height = 8, dpi = 300)

In [ ]:
p_sensi

In [ ]:
ggsave("sensitivityanalysis_gsk3b_adjustedmodels.png", p_sensi, width = 12, height = 10)